# Introduction to deep learning- Assignment 1


In [1]:
from tqdm.auto import tqdm,trange
import torch
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassF1Score
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torch.utils.data import Dataset
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import torch.nn.functional as F
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
print(torch.cuda.get_device_name(device)
)

Using cuda device
AMD Radeon RX 7600S


# Loading Training and Test sets from PyTorch

In [2]:
target_transform = v2.Lambda(
    lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
)
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

Trainig_data_ci = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform

)

test_data_ci = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

# MLP class using 2 hidden layers

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self, x = 0.0, y = 0.0, input = 28*28, first_hidden = 512,
                 second_hidden = 512, first_activation = nn.ReLU, second_activation = nn.ReLU):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_stack = nn.Sequential(
            nn.Linear(input, first_hidden),
            first_activation(),
            nn.Dropout(x),
            nn.Linear(first_hidden, second_hidden),
            second_activation(),
            nn.Dropout(y),
            nn.Linear(second_hidden, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_stack(x)
        return logits

# Weight initialization

In [4]:
def init_weights(model, weight_init=None, bias_init=None):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if weight_init is not None:
                weight_init(m.weight)
            if bias_init is not None:
                bias_init(m.bias)

# Training with MLP using Train and Validation split (with default ratio of 80/20)

In [5]:
def train_model(**kwargs):
    model         = kwargs.get("model", NeuralNetwork())
    train_dataset = kwargs.get("train_dataset", training_data)
    seed          = kwargs.get("seed", 528)
    epoches       = kwargs.get("epoches", 15)
    lr            = kwargs.get("lr", 0.003)
    optimizer_cls = kwargs.get("optimizer_cls", torch.optim.Adam)
    loss_FM       = kwargs.get("loss_fn", nn.CrossEntropyLoss())
    val_ratio     = kwargs.get("val_ratio", 0.2)
    batch_size    = kwargs.get("batch_size", 64)
    num_classes   = kwargs.get("num_classes", 10)
    average       = kwargs.get("average", "macro")
    save_path     = kwargs.get("save_path", "best_model.pt")
    dev           = kwargs.get("device", device)
    weight_init   = kwargs.get("weight_init", None)
    bias_init     = kwargs.get("bias_init", None)
    optimizer_kwargs = kwargs.get("optimizer_kwargs", {})

    history = {
        "train_loss": [], "train_acc": [], "train_prec": [], "train_f1": [],
        "val_loss": [],   "val_acc": [],   "val_prec": [],   "val_f1": [],
    }
    best_val_acc = 0.0
    acc  = MulticlassAccuracy(num_classes=num_classes).to(dev)
    prec = MulticlassPrecision(num_classes=num_classes, average=average).to(dev)
    f1   = MulticlassF1Score(num_classes=num_classes, average=average).to(dev)
    init_weights(model, weight_init, bias_init)
    optimizer = optimizer_cls(model.parameters(), lr=lr, **optimizer_kwargs)
    train_split, val_split = random_split(train_dataset, [len(train_dataset) - int(len(train_dataset) * val_ratio), int(len(train_dataset) * val_ratio)], generator=torch.Generator().manual_seed(seed))
    train_loader = DataLoader(dataset=train_split, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=val_split, batch_size=batch_size, shuffle=True)

    train_bar = tqdm(total=len(train_loader), desc="train")
    val_bar   = tqdm(total=len(val_loader),   desc="val")

    for epoch in trange(epoches, desc="Training"):

        train_loss_total = 0.0
        val_loss_total = 0.0
        model.train().to(dev)

        train_bar.reset()
        train_bar.set_description(f"epoch {epoch} train")
        for x, y in train_loader:
            x = x.to(dev)
            y = y.to(dev)
            logits = model(x)
            loss = loss_FM(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            preds = logits.argmax(dim=1)
            acc_v = acc(preds, y)
            prec_v = prec(preds, y)
            f1_v = f1(preds, y)
            train_loss_total += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}",
                                  prec=f"{prec_v:.4f}", f1=f"{f1_v:.4f}")
            train_bar.update(1)

        history["train_loss"].append(train_loss_total / len(train_loader))
        history["train_acc"].append(acc.compute().item())
        history["train_prec"].append(prec.compute().item())
        history["train_f1"].append(f1.compute().item())

        tqdm.write(f"epoch {epoch} train | loss {history['train_loss'][-1]:.4f} | "
                   f"Accuracy {history['train_acc'][-1]:.4f} | Precision {history['train_prec'][-1]:.4f} | "
                   f"F1 {history['train_f1'][-1]:.4f}")
        acc.reset()
        prec.reset()
        f1.reset()

        with torch.no_grad():
            model.eval()
            val_bar.reset()
            val_bar.set_description(f"epoch {epoch} val")
            for x, y in val_loader:
                x = x.to(dev)
                y = y.to(dev)
                logits = model(x)
                loss = loss_FM(logits, y)
                preds = logits.argmax(dim=1)
                acc_v = acc(preds, y)
                prec_v = prec(preds, y)
                f1_v = f1(preds, y)
                val_loss_total += loss.item()
                val_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}")
                val_bar.update(1)

        history["val_loss"].append(val_loss_total / len(val_loader))
        history["val_acc"].append(acc.compute().item())
        history["val_prec"].append(prec.compute().item())
        history["val_f1"].append(f1.compute().item())

        tqdm.write(f"epoch {epoch} val   | loss {history['val_loss'][-1]:.4f} | "
                   f"Accuracy {history['val_acc'][-1]:.4f} | Precision {history['val_prec'][-1]:.4f} | "
                   f"F1 {history['val_f1'][-1]:.4f}")
        if history["val_acc"][-1] > best_val_acc:
            best_val_acc = history["val_acc"][-1]
            torch.save(model.state_dict(), save_path)
            tqdm.write(f"new best val acc {best_val_acc:.4f}, saved")

        acc.reset()
        prec.reset()
        f1.reset()

    train_bar.close()
    val_bar.close()

    return history, best_val_acc

# Evaluation function to be used with test set

In [6]:
def evaluate_model(**kwargs):
    test_model   = kwargs.get("model", NeuralNetwork())
    test_dataset = kwargs.get("test_dataset", test_data)
    load_path    = kwargs.get("load_path", "best_model.pt")
    batch_size   = kwargs.get("batch_size", 64)
    loss_FM      = kwargs.get("loss_fn", nn.CrossEntropyLoss())
    num_classes  = kwargs.get("num_classes", 10)
    average      = kwargs.get("average", "macro")
    dev          = kwargs.get("device", device)

    acc  = MulticlassAccuracy(num_classes=num_classes).to(dev)
    prec = MulticlassPrecision(num_classes=num_classes, average=average).to(dev)
    f1   = MulticlassF1Score(num_classes=num_classes, average=average).to(dev)

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    test_model.load_state_dict(torch.load(load_path, map_location=dev))
    test_model.to(dev)
    test_model.eval()

    acc.reset(); prec.reset(); f1.reset()
    test_loss_total = 0.0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(dev), y.to(dev)
            logits = test_model(x)
            test_loss_total += loss_FM(logits, y).item()
            preds = logits.argmax(dim=1)
            acc.update(preds, y)
            prec.update(preds, y)
            f1.update(preds, y)

    print(f"test | loss {test_loss_total/len(test_loader):.4f} | "
          f"Accuracy {acc.compute():.4f} | Precision {prec.compute():.4f} | F1 {f1.compute():.4f}")

    return {
        "loss": test_loss_total / len(test_loader),
        "acc": acc.compute().item(),
        "prec": prec.compute().item(),
        "f1": f1.compute().item(),
    }

# Run 1 without dropout (FashionMNIST)

In [7]:
# Run 1: no dropout
history, best = train_model(model=NeuralNetwork(), save_path="best_model.pt")
evaluate_model(model=NeuralNetwork(), load_path="best_model.pt")



train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5053 | Accuracy 0.8142 | Precision 0.8124 | F1 0.8121
epoch 0 val   | loss 0.4420 | Accuracy 0.8454 | Precision 0.8510 | F1 0.8457
new best val acc 0.8454, saved
epoch 1 train | loss 0.3808 | Accuracy 0.8609 | Precision 0.8604 | F1 0.8603
epoch 1 val   | loss 0.3739 | Accuracy 0.8659 | Precision 0.8707 | F1 0.8668
new best val acc 0.8659, saved
epoch 2 train | loss 0.3452 | Accuracy 0.8738 | Precision 0.8733 | F1 0.8733
epoch 2 val   | loss 0.3910 | Accuracy 0.8620 | Precision 0.8680 | F1 0.8588
epoch 3 train | loss 0.3203 | Accuracy 0.8820 | Precision 0.8816 | F1 0.8815
epoch 3 val   | loss 0.3917 | Accuracy 0.8645 | Precision 0.8711 | F1 0.8661
epoch 4 train | loss 0.3070 | Accuracy 0.8852 | Precision 0.8849 | F1 0.8848
epoch 4 val   | loss 0.3456 | Accuracy 0.8777 | Precision 0.8789 | F1 0.8757
new best val acc 0.8777, saved
epoch 5 train | loss 0.2915 | Accuracy 0.8924 | Precision 0.8922 | F1 0.8921
epoch 5 val   | loss 0.3570 | Accuracy 0.8736 | Precision 0.

{'loss': 0.3620406173311981,
 'acc': 0.8837000131607056,
 'prec': 0.8839738368988037,
 'f1': 0.8836844563484192}

# Run 2 with dropout 0.2 (FashionMNIST)

In [8]:
# Run 2: dropout 0.2 / 0.2
history_drop, best_drop = train_model(model=NeuralNetwork(x=0.2, y=0.2), save_path="best_model_drop.pt")
evaluate_model(model=NeuralNetwork(x=0.2, y=0.2), load_path="best_model_drop.pt")

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5559 | Accuracy 0.7959 | Precision 0.7939 | F1 0.7935
epoch 0 val   | loss 0.4441 | Accuracy 0.8379 | Precision 0.8434 | F1 0.8374
new best val acc 0.8379, saved
epoch 1 train | loss 0.4405 | Accuracy 0.8399 | Precision 0.8389 | F1 0.8389
epoch 1 val   | loss 0.4027 | Accuracy 0.8520 | Precision 0.8556 | F1 0.8499
new best val acc 0.8520, saved
epoch 2 train | loss 0.4195 | Accuracy 0.8470 | Precision 0.8461 | F1 0.8461
epoch 2 val   | loss 0.3784 | Accuracy 0.8636 | Precision 0.8635 | F1 0.8625
new best val acc 0.8636, saved
epoch 3 train | loss 0.3999 | Accuracy 0.8534 | Precision 0.8524 | F1 0.8525
epoch 3 val   | loss 0.3769 | Accuracy 0.8657 | Precision 0.8702 | F1 0.8671
new best val acc 0.8657, saved
epoch 4 train | loss 0.3824 | Accuracy 0.8609 | Precision 0.8599 | F1 0.8601
epoch 4 val   | loss 0.3824 | Accuracy 0.8626 | Precision 0.8684 | F1 0.8638
epoch 5 train | loss 0.3794 | Accuracy 0.8617 | Precision 0.8610 | F1 0.8610
epoch 5 val   | loss 0.3684 |

{'loss': 0.3656728072151257,
 'acc': 0.8716000318527222,
 'prec': 0.8707919120788574,
 'f1': 0.8702991008758545}

# Function to run experiments for different parameters

In [9]:
results = {}

def run_experiment(experiment, runs, model_cls=NeuralNetwork, init_seed=528):
    for run in runs:
        name = f"{experiment}_{run['name']}"
        model_kwargs = run.get("model_kwargs", {})
        train_kwargs = run.get("train_kwargs", {})

        print(f"\n===== {name} =====")
        torch.manual_seed(init_seed)
        model = model_cls(**model_kwargs)
        history, best = train_model(model=model, save_path=f"{name}.pt", **train_kwargs)
        test = evaluate_model(model=model_cls(**model_kwargs), load_path=f"{name}.pt")

        results[name] = {"history": history, "best_val_acc": best, "test": test}

# Parameter initialization (torch.nn.init)


In [10]:
run_experiment("init", [
    {"name": "default"},
    {"name": "kaiming_uniform", "train_kwargs": {
        "weight_init": lambda w:torch.nn.init.kaiming_uniform_(w, mode='fan_in', nonlinearity='relu'),
        "bias_init":   lambda b: torch.nn.init.constant_(b, 0.1)}},
    {"name": "uni", "train_kwargs": {"weight_init": lambda w:torch.nn.init.uniform_(w, a=0.0, b=1.0),
                                     "bias_init": lambda b: torch.nn.init.constant_(b, 0)}},
    {"name": "uni_biasnonzero", "train_kwargs": {"weight_init": lambda w:torch.nn.init.uniform_(w, a=0.0, b=1.0),
                                     "bias_init": lambda b: torch.nn.init.constant_(b, 0.1)}},
])


===== init_default =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5063 | Accuracy 0.8146 | Precision 0.8136 | F1 0.8131
epoch 0 val   | loss 0.4163 | Accuracy 0.8469 | Precision 0.8593 | F1 0.8501
new best val acc 0.8469, saved
epoch 1 train | loss 0.3828 | Accuracy 0.8627 | Precision 0.8624 | F1 0.8622
epoch 1 val   | loss 0.3766 | Accuracy 0.8651 | Precision 0.8684 | F1 0.8625
new best val acc 0.8651, saved
epoch 2 train | loss 0.3430 | Accuracy 0.8738 | Precision 0.8733 | F1 0.8733
epoch 2 val   | loss 0.3653 | Accuracy 0.8702 | Precision 0.8728 | F1 0.8680
new best val acc 0.8702, saved
epoch 3 train | loss 0.3216 | Accuracy 0.8813 | Precision 0.8809 | F1 0.8809
epoch 3 val   | loss 0.3583 | Accuracy 0.8714 | Precision 0.8743 | F1 0.8713
new best val acc 0.8714, saved
epoch 4 train | loss 0.3079 | Accuracy 0.8852 | Precision 0.8848 | F1 0.8848
epoch 4 val   | loss 0.3510 | Accuracy 0.8717 | Precision 0.8738 | F1 0.8718
new best val acc 0.8717, saved
epoch 5 train | loss 0.2925 | Accuracy 0.8907 | Precision 0.8904 | F1 0.890

...

epoch 0 train | loss 7706.8702 | Accuracy 0.4243 | Precision 0.4283 | F1 0.4240
epoch 0 val   | loss 2524.3623 | Accuracy 0.5181 | Precision 0.5503 | F1 0.4618
new best val acc 0.5181, saved
epoch 1 train | loss 794.8559 | Accuracy 0.6723 | Precision 0.6707 | F1 0.6709
epoch 1 val   | loss 413.3469 | Accuracy 0.7257 | Precision 0.7683 | F1 0.7102
new best val acc 0.7257, saved
epoch 2 train | loss 430.6646 | Accuracy 0.7462 | Precision 0.7452 | F1 0.7455
epoch 2 val   | loss 166.9701 | Accuracy 0.8130 | Precision 0.8258 | F1 0.8163
new best val acc 0.8130, saved
epoch 3 train | loss 218.4186 | Accuracy 0.7868 | Precision 0.7866 | F1 0.7866
epoch 3 val   | loss 148.4275 | Accuracy 0.7828 | Precision 0.8141 | F1 0.7778
epoch 4 train | loss 98.9000 | Accuracy 0.8058 | Precision 0.8058 | F1 0.8057
epoch 4 val   | loss 65.0532 | Accuracy 0.8114 | Precision 0.8429 | F1 0.8107
epoch 5 train | loss 43.8470 | Accuracy 0.8184 | Precision 0.8186 | F1 0.8185
epoch 5 val   | loss 30.3094 | Accuracy

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 7555.1737 | Accuracy 0.4200 | Precision 0.4231 | F1 0.4192
epoch 0 val   | loss 2081.6004 | Accuracy 0.5743 | Precision 0.6700 | F1 0.5246
new best val acc 0.5743, saved
epoch 1 train | loss 858.4639 | Accuracy 0.6691 | Precision 0.6674 | F1 0.6676
epoch 1 val   | loss 446.1501 | Accuracy 0.7378 | Precision 0.8203 | F1 0.7390
new best val acc 0.7378, saved
epoch 2 train | loss 416.6943 | Accuracy 0.7512 | Precision 0.7502 | F1 0.7505
epoch 2 val   | loss 189.5769 | Accuracy 0.8245 | Precision 0.8329 | F1 0.8226
new best val acc 0.8245, saved
epoch 3 train | loss 218.2636 | Accuracy 0.7879 | Precision 0.7877 | F1 0.7877
epoch 3 val   | loss 162.0470 | Accuracy 0.7717 | Precision 0.8119 | F1 0.7678
epoch 4 train | loss 104.4213 | Accuracy 0.8017 | Precision 0.8017 | F1 0.8017
epoch 4 val   | loss 62.2519 | Accuracy 0.8210 | Precision 0.8393 | F1 0.8210
epoch 5 train | loss 44.3059 | Accuracy 0.8193 | Precision 0.8195 | F1 0.8194
epoch 5 val   | loss 36.3599 | Accurac

# Activation functions

In [11]:

run_experiment("activation", [
    {"name": "relu"},
    {"name": "LeakyReLU", "model_kwargs": {"first_activation": nn.LeakyReLU, "second_activation": nn.LeakyReLU}},
    {"name": "nn.GELU", "model_kwargs": {"first_activation": nn.GELU, "second_activation": nn.GELU}},
    {"name": "combination1", "model_kwargs": {"first_activation": nn.ReLU, "second_activation": nn.GELU}},
    {"name": "combination2", "model_kwargs": {"first_activation": nn.GELU, "second_activation": nn.LeakyReLU}},
])


===== activation_relu =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5063 | Accuracy 0.8146 | Precision 0.8136 | F1 0.8131
epoch 0 val   | loss 0.4163 | Accuracy 0.8469 | Precision 0.8593 | F1 0.8501
new best val acc 0.8469, saved
epoch 1 train | loss 0.3828 | Accuracy 0.8627 | Precision 0.8624 | F1 0.8622
epoch 1 val   | loss 0.3766 | Accuracy 0.8651 | Precision 0.8684 | F1 0.8625
new best val acc 0.8651, saved
epoch 2 train | loss 0.3430 | Accuracy 0.8738 | Precision 0.8733 | F1 0.8733
epoch 2 val   | loss 0.3653 | Accuracy 0.8702 | Precision 0.8728 | F1 0.8680
new best val acc 0.8702, saved
epoch 3 train | loss 0.3216 | Accuracy 0.8813 | Precision 0.8809 | F1 0.8809
epoch 3 val   | loss 0.3583 | Accuracy 0.8714 | Precision 0.8743 | F1 0.8713
new best val acc 0.8714, saved
epoch 4 train | loss 0.3079 | Accuracy 0.8852 | Precision 0.8848 | F1 0.8848
epoch 4 val   | loss 0.3510 | Accuracy 0.8717 | Precision 0.8738 | F1 0.8718
new best val acc 0.8717, saved
epoch 5 train | loss 0.2925 | Accuracy 0.8907 | Precision 0.8904 | F1 0.890

...

epoch 0 train | loss 0.5031 | Accuracy 0.8146 | Precision 0.8128 | F1 0.8130
epoch 0 val   | loss 0.4284 | Accuracy 0.8398 | Precision 0.8608 | F1 0.8428
new best val acc 0.8398, saved
epoch 1 train | loss 0.3801 | Accuracy 0.8610 | Precision 0.8604 | F1 0.8604
epoch 1 val   | loss 0.3747 | Accuracy 0.8677 | Precision 0.8695 | F1 0.8642
new best val acc 0.8677, saved
epoch 2 train | loss 0.3445 | Accuracy 0.8738 | Precision 0.8735 | F1 0.8734
epoch 2 val   | loss 0.3534 | Accuracy 0.8703 | Precision 0.8711 | F1 0.8691
new best val acc 0.8703, saved
epoch 3 train | loss 0.3213 | Accuracy 0.8809 | Precision 0.8804 | F1 0.8804
epoch 3 val   | loss 0.3438 | Accuracy 0.8722 | Precision 0.8739 | F1 0.8719
new best val acc 0.8722, saved
epoch 4 train | loss 0.3054 | Accuracy 0.8869 | Precision 0.8865 | F1 0.8866
epoch 4 val   | loss 0.3387 | Accuracy 0.8747 | Precision 0.8758 | F1 0.8734
new best val acc 0.8747, saved
epoch 5 train | loss 0.2913 | Accuracy 0.8912 | Precision 0.8907 | F1 0.890

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.4931 | Accuracy 0.8172 | Precision 0.8154 | F1 0.8152
epoch 0 val   | loss 0.4324 | Accuracy 0.8359 | Precision 0.8579 | F1 0.8383
new best val acc 0.8359, saved
epoch 1 train | loss 0.3736 | Accuracy 0.8648 | Precision 0.8643 | F1 0.8641
epoch 1 val   | loss 0.3670 | Accuracy 0.8631 | Precision 0.8668 | F1 0.8609
new best val acc 0.8631, saved
epoch 2 train | loss 0.3415 | Accuracy 0.8750 | Precision 0.8746 | F1 0.8745
epoch 2 val   | loss 0.3584 | Accuracy 0.8718 | Precision 0.8715 | F1 0.8700
new best val acc 0.8718, saved
epoch 3 train | loss 0.3126 | Accuracy 0.8828 | Precision 0.8826 | F1 0.8824
epoch 3 val   | loss 0.4016 | Accuracy 0.8593 | Precision 0.8630 | F1 0.8572
epoch 4 train | loss 0.3012 | Accuracy 0.8880 | Precision 0.8875 | F1 0.8874
epoch 4 val   | loss 0.3571 | Accuracy 0.8672 | Precision 0.8687 | F1 0.8663
epoch 5 train | loss 0.2872 | Accuracy 0.8942 | Precision 0.8939 | F1 0.8938
epoch 5 val   | loss 0.3747 | Accuracy 0.8620 | Precision 0.

# Optimizers

In [12]:
run_experiment("optimizer", [
    {"name": "SGD", "train_kwargs": {"optimizer_cls": torch.optim.SGD, "lr": 0.005,
                                     "optimizer_kwargs": {"momentum": 0.89}}},
    {"name": "AdamW", "train_kwargs": {"optimizer_cls": torch.optim.AdamW, "lr": 0.005,
                                       "optimizer_kwargs": {"weight_decay": 0.05}}},
    {"name": "Adam", "train_kwargs": {"optimizer_cls": torch.optim.Adam, "lr": 0.005}},
])


===== optimizer_SGD =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.8566 | Accuracy 0.7034 | Precision 0.7149 | F1 0.6989
epoch 0 val   | loss 0.5424 | Accuracy 0.8113 | Precision 0.8237 | F1 0.8135
new best val acc 0.8113, saved
epoch 1 train | loss 0.4933 | Accuracy 0.8266 | Precision 0.8244 | F1 0.8249
epoch 1 val   | loss 0.4556 | Accuracy 0.8457 | Precision 0.8456 | F1 0.8428
new best val acc 0.8457, saved
epoch 2 train | loss 0.4374 | Accuracy 0.8443 | Precision 0.8429 | F1 0.8432
epoch 2 val   | loss 0.4325 | Accuracy 0.8504 | Precision 0.8531 | F1 0.8464
new best val acc 0.8504, saved
epoch 3 train | loss 0.4020 | Accuracy 0.8576 | Precision 0.8564 | F1 0.8567
epoch 3 val   | loss 0.3957 | Accuracy 0.8609 | Precision 0.8607 | F1 0.8603
new best val acc 0.8609, saved
epoch 4 train | loss 0.3804 | Accuracy 0.8647 | Precision 0.8636 | F1 0.8639
epoch 4 val   | loss 0.3906 | Accuracy 0.8626 | Precision 0.8645 | F1 0.8615
new best val acc 0.8626, saved
epoch 5 train | loss 0.3600 | Accuracy 0.8716 | Precision 0.8707 | F1 0.870

...

epoch 0 train | loss 0.5236 | Accuracy 0.8072 | Precision 0.8056 | F1 0.8052
epoch 0 val   | loss 0.4493 | Accuracy 0.8425 | Precision 0.8473 | F1 0.8423
new best val acc 0.8425, saved
epoch 1 train | loss 0.4083 | Accuracy 0.8525 | Precision 0.8517 | F1 0.8516
epoch 1 val   | loss 0.3882 | Accuracy 0.8587 | Precision 0.8610 | F1 0.8562
new best val acc 0.8587, saved
epoch 2 train | loss 0.3755 | Accuracy 0.8628 | Precision 0.8621 | F1 0.8621
epoch 2 val   | loss 0.3925 | Accuracy 0.8587 | Precision 0.8637 | F1 0.8571
new best val acc 0.8587, saved
epoch 3 train | loss 0.3642 | Accuracy 0.8660 | Precision 0.8655 | F1 0.8654
epoch 3 val   | loss 0.3733 | Accuracy 0.8661 | Precision 0.8646 | F1 0.8634
new best val acc 0.8661, saved
epoch 4 train | loss 0.3504 | Accuracy 0.8724 | Precision 0.8717 | F1 0.8717
epoch 4 val   | loss 0.3853 | Accuracy 0.8587 | Precision 0.8612 | F1 0.8585
epoch 5 train | loss 0.3460 | Accuracy 0.8739 | Precision 0.8733 | F1 0.8732
epoch 5 val   | loss 0.3974 |

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5215 | Accuracy 0.8102 | Precision 0.8085 | F1 0.8083
epoch 0 val   | loss 0.4134 | Accuracy 0.8485 | Precision 0.8582 | F1 0.8511
new best val acc 0.8485, saved
epoch 1 train | loss 0.3976 | Accuracy 0.8572 | Precision 0.8564 | F1 0.8564
epoch 1 val   | loss 0.4125 | Accuracy 0.8545 | Precision 0.8585 | F1 0.8509
new best val acc 0.8545, saved
epoch 2 train | loss 0.3603 | Accuracy 0.8692 | Precision 0.8687 | F1 0.8687
epoch 2 val   | loss 0.3628 | Accuracy 0.8703 | Precision 0.8713 | F1 0.8702
new best val acc 0.8703, saved
epoch 3 train | loss 0.3405 | Accuracy 0.8755 | Precision 0.8749 | F1 0.8750
epoch 3 val   | loss 0.3592 | Accuracy 0.8706 | Precision 0.8695 | F1 0.8690
new best val acc 0.8706, saved
epoch 4 train | loss 0.3296 | Accuracy 0.8791 | Precision 0.8786 | F1 0.8786
epoch 4 val   | loss 0.3665 | Accuracy 0.8644 | Precision 0.8666 | F1 0.8645
epoch 5 train | loss 0.3191 | Accuracy 0.8827 | Precision 0.8822 | F1 0.8822
epoch 5 val   | loss 0.4117 |

# Architecture: width

In [13]:
run_experiment("width", [
    {"name": "128", "model_kwargs": {"first_hidden": 128, "second_hidden": 128}},
    {"name": "256", "model_kwargs": {"first_hidden": 256, "second_hidden": 256}},
    {"name": "784", "model_kwargs": {"first_hidden": 784, "second_hidden": 784}},
])


===== width_128 =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5260 | Accuracy 0.8062 | Precision 0.8049 | F1 0.8044
epoch 0 val   | loss 0.4043 | Accuracy 0.8541 | Precision 0.8568 | F1 0.8511
new best val acc 0.8541, saved
epoch 1 train | loss 0.3830 | Accuracy 0.8595 | Precision 0.8588 | F1 0.8588
epoch 1 val   | loss 0.4194 | Accuracy 0.8471 | Precision 0.8669 | F1 0.8426
epoch 2 train | loss 0.3449 | Accuracy 0.8740 | Precision 0.8735 | F1 0.8735
epoch 2 val   | loss 0.3628 | Accuracy 0.8716 | Precision 0.8747 | F1 0.8689
new best val acc 0.8716, saved
epoch 3 train | loss 0.3240 | Accuracy 0.8806 | Precision 0.8801 | F1 0.8801
epoch 3 val   | loss 0.3674 | Accuracy 0.8635 | Precision 0.8682 | F1 0.8632
epoch 4 train | loss 0.3080 | Accuracy 0.8846 | Precision 0.8841 | F1 0.8842
epoch 4 val   | loss 0.3421 | Accuracy 0.8724 | Precision 0.8738 | F1 0.8708
new best val acc 0.8724, saved
epoch 5 train | loss 0.2940 | Accuracy 0.8899 | Precision 0.8895 | F1 0.8895
epoch 5 val   | loss 0.3597 | Accuracy 0.8729 | Precision 0.

...

epoch 0 train | loss 0.5152 | Accuracy 0.8122 | Precision 0.8106 | F1 0.8105
epoch 0 val   | loss 0.4199 | Accuracy 0.8424 | Precision 0.8475 | F1 0.8425
new best val acc 0.8424, saved
epoch 1 train | loss 0.3847 | Accuracy 0.8592 | Precision 0.8586 | F1 0.8585
epoch 1 val   | loss 0.3807 | Accuracy 0.8646 | Precision 0.8661 | F1 0.8624
new best val acc 0.8646, saved
epoch 2 train | loss 0.3473 | Accuracy 0.8730 | Precision 0.8725 | F1 0.8725
epoch 2 val   | loss 0.3491 | Accuracy 0.8702 | Precision 0.8697 | F1 0.8684
new best val acc 0.8702, saved
epoch 3 train | loss 0.3199 | Accuracy 0.8815 | Precision 0.8813 | F1 0.8811
epoch 3 val   | loss 0.3514 | Accuracy 0.8751 | Precision 0.8754 | F1 0.8742
new best val acc 0.8751, saved
epoch 4 train | loss 0.3039 | Accuracy 0.8880 | Precision 0.8877 | F1 0.8877
epoch 4 val   | loss 0.3489 | Accuracy 0.8740 | Precision 0.8772 | F1 0.8734
epoch 5 train | loss 0.2908 | Accuracy 0.8908 | Precision 0.8904 | F1 0.8904
epoch 5 val   | loss 0.3527 |

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5107 | Accuracy 0.8144 | Precision 0.8130 | F1 0.8127
epoch 0 val   | loss 0.4126 | Accuracy 0.8473 | Precision 0.8495 | F1 0.8447
new best val acc 0.8473, saved
epoch 1 train | loss 0.3839 | Accuracy 0.8592 | Precision 0.8584 | F1 0.8585
epoch 1 val   | loss 0.3898 | Accuracy 0.8578 | Precision 0.8589 | F1 0.8559
new best val acc 0.8578, saved
epoch 2 train | loss 0.3412 | Accuracy 0.8746 | Precision 0.8741 | F1 0.8741
epoch 2 val   | loss 0.3492 | Accuracy 0.8742 | Precision 0.8742 | F1 0.8725
new best val acc 0.8742, saved
epoch 3 train | loss 0.3228 | Accuracy 0.8801 | Precision 0.8796 | F1 0.8796
epoch 3 val   | loss 0.3947 | Accuracy 0.8601 | Precision 0.8670 | F1 0.8605
epoch 4 train | loss 0.2999 | Accuracy 0.8889 | Precision 0.8887 | F1 0.8886
epoch 4 val   | loss 0.3590 | Accuracy 0.8712 | Precision 0.8712 | F1 0.8679
epoch 5 train | loss 0.2882 | Accuracy 0.8920 | Precision 0.8917 | F1 0.8916
epoch 5 val   | loss 0.3413 | Accuracy 0.8797 | Precision 0.

In [14]:
for name, r in results.items():
    t = r["test"]
    print(f"{name:30s} | best val acc {r['best_val_acc']:.4f} | test loss {t['loss']:.4f} | "
          f"test acc {t['acc']:.4f} | prec {t['prec']:.4f} | f1 {t['f1']:.4f}")

init_default                   | best val acc 0.8847 | test loss 0.3781 | test acc 0.8781 | prec 0.8778 | f1 0.8774
init_kaiming_uniform           | best val acc 0.8864 | test loss 0.3801 | test acc 0.8830 | prec 0.8828 | f1 0.8821
init_uni                       | best val acc 0.8420 | test loss 1.0565 | test acc 0.8334 | prec 0.8339 | f1 0.8293
init_uni_biasnonzero           | best val acc 0.8436 | test loss 0.8297 | test acc 0.8350 | prec 0.8377 | f1 0.8357
activation_relu                | best val acc 0.8847 | test loss 0.3781 | test acc 0.8781 | prec 0.8778 | f1 0.8774
activation_LeakyReLU           | best val acc 0.8825 | test loss 0.3644 | test acc 0.8840 | prec 0.8834 | f1 0.8824
activation_nn.GELU             | best val acc 0.8878 | test loss 0.3625 | test acc 0.8835 | prec 0.8840 | f1 0.8834
activation_combination1        | best val acc 0.8867 | test loss 0.3855 | test acc 0.8819 | prec 0.8816 | f1 0.8809
activation_combination2        | best val acc 0.8803 | test loss 0.3690 